# Checkstyle — Java Coding Standard Enforcement

[Checkstyle](https://checkstyle.sourceforge.io/) validates Java source code against a configurable coding standard.

## What Checkstyle produces
- **Violations** with: severity (`ERROR`/`WARN`/`INFO`), file, line, column, rule class, message.
- Formats: `plain` (default), `xml`, `sarif`.
- Built-in configs: **`/google_checks.xml`**, **`/sun_checks.xml`** (bundled in the all-JAR).

## CLI
```
java -jar checkstyle-all.jar -c <config.xml> [-f <format>] [-o <report-file>] <source-path>
```

## 1. Configuration

In [1]:
from __future__ import annotations
import os, shutil, subprocess, sys
from pathlib import Path

_base = Path(r"F:\java_metrics")
PROJECT_ROOT      = Path(os.environ.get("JAVA_PROJECT_ROOT", _base / "sample-java-app" / "src")).resolve()
CHECKSTYLE_JAR    = Path(os.environ.get("CHECKSTYLE_JAR",    _base / "tools" / "checkstyle-all.jar")).resolve()
OUTPUT_DIR        = _base / "checkstyle_out"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve real java.exe (avoid Windows System32 stub)
def _find_java() -> str:
    if jh := os.environ.get("JAVA_HOME"):
        p = Path(jh) / "bin" / ("java.exe" if sys.platform == "win32" else "java")
        if p.is_file(): return str(p)
    for candidate in (shutil.which("java") or "").splitlines():
        if "System32" not in candidate: return candidate
    return shutil.which("java") or "java"
JAVA_EXE = _find_java()

# Built-in configs: /google_checks.xml  or  /sun_checks.xml
CHECKSTYLE_CONFIG = os.environ.get("CHECKSTYLE_CONFIG", "/google_checks.xml")
FORMATS = ["plain", "xml", "sarif"]

assert CHECKSTYLE_JAR.is_file(),  f"checkstyle-all.jar not found: {CHECKSTYLE_JAR}"
assert PROJECT_ROOT.is_dir(),     f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print(f"JAVA_EXE          : {JAVA_EXE}")
print(f"CHECKSTYLE_JAR    : {CHECKSTYLE_JAR}")
print(f"CHECKSTYLE_CONFIG : {CHECKSTYLE_CONFIG}")
print(f"PROJECT_ROOT      : {PROJECT_ROOT}")
print(f"OUTPUT_DIR        : {OUTPUT_DIR}")

JAVA_EXE          : C:\Program Files\Java\jdk-21.0.10\bin\java.EXE
CHECKSTYLE_JAR    : F:\java_metrics\tools\checkstyle-all.jar
CHECKSTYLE_CONFIG : /google_checks.xml
PROJECT_ROOT      : F:\java_metrics\sample-java-app\src
OUTPUT_DIR        : F:\java_metrics\checkstyle_out


## 2. Show Checkstyle version

In [2]:
proc = subprocess.run(
    [JAVA_EXE, "-jar", str(CHECKSTYLE_JAR), "--version"],
    capture_output=True, text=True
)
print(proc.stdout or proc.stderr)

Checkstyle version: 13.3.0



## 3. List Java source files

In [3]:
java_files = sorted(PROJECT_ROOT.rglob("*.java"))
print(f"Java files found: {len(java_files)}")
for f in java_files:
    print(" ", f.relative_to(PROJECT_ROOT))

Java files found: 1
  main\java\com\example\App.java


## 4. Run Checkstyle — raw output (plain, xml, sarif)

In [4]:
results = {}

for fmt in FORMATS:
    ext = {"plain": "txt", "xml": "xml", "sarif": "sarif.json"}.get(fmt, fmt)
    report_file = OUTPUT_DIR / f"checkstyle-report.{ext}"
    cmd = [
        JAVA_EXE, "-jar", str(CHECKSTYLE_JAR),
        "-c", CHECKSTYLE_CONFIG,
        "-f", fmt,
        "-o", str(report_file),
        str(PROJECT_ROOT),
    ]
    print(f"\n{'='*60}")
    print(f"Running Checkstyle [{fmt}]: {' '.join(cmd)}")
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print("STDOUT:", proc.stdout or "(empty)")
    print("STDERR:", proc.stderr or "(empty)")
    # exit 0 = no violations, 1 = violations, 2 = error
    print("Exit code:", proc.returncode,
          "(0 = clean, 1 = violations, 2 = error)")
    results[fmt] = report_file


Running Checkstyle [plain]: C:\Program Files\Java\jdk-21.0.10\bin\java.EXE -jar F:\java_metrics\tools\checkstyle-all.jar -c /google_checks.xml -f plain -o F:\java_metrics\checkstyle_out\checkstyle-report.txt F:\java_metrics\sample-java-app\src


STDOUT: (empty)
STDERR: (empty)
Exit code: 0 (0 = clean, 1 = violations, 2 = error)

Running Checkstyle [xml]: C:\Program Files\Java\jdk-21.0.10\bin\java.EXE -jar F:\java_metrics\tools\checkstyle-all.jar -c /google_checks.xml -f xml -o F:\java_metrics\checkstyle_out\checkstyle-report.xml F:\java_metrics\sample-java-app\src


STDOUT: (empty)
STDERR: (empty)
Exit code: 0 (0 = clean, 1 = violations, 2 = error)

Running Checkstyle [sarif]: C:\Program Files\Java\jdk-21.0.10\bin\java.EXE -jar F:\java_metrics\tools\checkstyle-all.jar -c /google_checks.xml -f sarif -o F:\java_metrics\checkstyle_out\checkstyle-report.sarif.json F:\java_metrics\sample-java-app\src


STDOUT: (empty)
STDERR: [main] INFO org.reflections.Reflections - Reflections took 48 ms to scan 1 urls, producing 213 keys and 213 values

Exit code: 0 (0 = clean, 1 = violations, 2 = error)


## 5. Raw plain-text report

In [5]:
txt = results["plain"]
if txt.is_file():
    print(f"=== {txt.name} ===")
    print(txt.read_text(encoding="utf-8", errors="replace") or "(empty — no violations)")
else:
    print("Plain report not produced.")

=== checkstyle-report.txt ===
Starting audit...
[WARN] F:\java_metrics\sample-java-app\src\main\java\com\example\App.java:3:1: Missing a Javadoc comment. [MissingJavadocType]
[WARN] F:\java_metrics\sample-java-app\src\main\java\com\example\App.java:4:5: 'method def modifier' has incorrect indentation level 4, expected level should be 2. [Indentation]
[WARN] F:\java_metrics\sample-java-app\src\main\java\com\example\App.java:4:5: Missing a Javadoc comment. [MissingJavadocMethod]
[WARN] F:\java_metrics\sample-java-app\src\main\java\com\example\App.java:5:9: 'method def' child has incorrect indentation level 8, expected level should be 4. [Indentation]
[WARN] F:\java_metrics\sample-java-app\src\main\java\com\example\App.java:6:5: 'method def rcurly' has incorrect indentation level 4, expected level should be 2. [Indentation]
Audit done.



## 6. Raw XML report

In [6]:
xml = results["xml"]
if xml.is_file():
    print(f"=== {xml.name} ===")
    print(xml.read_text(encoding="utf-8", errors="replace"))
else:
    print("XML report not produced.")

=== checkstyle-report.xml ===
<?xml version="1.0" encoding="UTF-8"?>
<checkstyle version="13.3.0">
<file name="F:\java_metrics\sample-java-app\src\main\java\com\example\App.java">
<error line="3" column="1" severity="warning" message="Missing a Javadoc comment." source="com.puppycrawl.tools.checkstyle.checks.javadoc.MissingJavadocTypeCheck"/>
<error line="4" column="5" severity="warning" message="&apos;method def modifier&apos; has incorrect indentation level 4, expected level should be 2." source="com.puppycrawl.tools.checkstyle.checks.indentation.IndentationCheck"/>
<error line="4" column="5" severity="warning" message="Missing a Javadoc comment." source="com.puppycrawl.tools.checkstyle.checks.javadoc.MissingJavadocMethodCheck"/>
<error line="5" column="9" severity="warning" message="&apos;method def&apos; child has incorrect indentation level 8, expected level should be 4." source="com.puppycrawl.tools.checkstyle.checks.indentation.IndentationCheck"/>
<error line="6" column="5" seve

## 7. Raw SARIF report

In [7]:
import json as _json
sarif = results["sarif"]
if sarif.is_file():
    raw = sarif.read_text(encoding="utf-8", errors="replace")
    print(f"=== {sarif.name} ===")
    try:
        print(_json.dumps(_json.loads(raw), indent=2))
    except Exception:
        print(raw)
else:
    print("SARIF report not produced.")

=== checkstyle-report.sarif.json ===
{
  "$schema": "https://raw.githubusercontent.com/oasis-tcs/sarif-spec/master/Schemata/sarif-schema-2.1.0.json",
  "version": "2.1.0",
  "runs": [
    {
      "tool": {
        "driver": {
          "downloadUri": "https://github.com/checkstyle/checkstyle/releases/",
          "fullName": "Checkstyle",
          "informationUri": "https://checkstyle.org/",
          "language": "en",
          "name": "Checkstyle",
          "organization": "Checkstyle",
          "rules": [
            {
              "id": "com.puppycrawl.tools.checkstyle.checks.javadoc.MissingJavadocTypeCheck",
              "messageStrings": {
                "javadoc.missing": {
                  "text": "Missing a Javadoc comment."
                }
              },
              "shortDescription": {
                "text": "MissingJavadocType"
              },
              "fullDescription": {
                "text": "<div>\n Checks for missing Javadoc comments for class, e

## 8. All output files

In [8]:
print("Files in output directory:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

Files in output directory:
  checkstyle-report.sarif.json  (9,361 bytes)
  checkstyle-report.txt  (822 bytes)
  checkstyle-report.xml  (1,199 bytes)
